## Case Study 6.2 - Resume Sorting

In this case study we will build a language model pipeline to extract key data points from resumes and then sort them according to a selection criteria.

This notebook is built from the scripts developed in the book:

* [Extract Text - CaseStudy_6.2_01-extract_text.py](CaseStudy_6.2_01-extract_text.py)
* [Extract Selection Criteria - CaseStudy_6.2_01-02.py](CaseStudy_6.2_01-02.py)
* [Evaluate Candidates - CaseStudy_6.2_01-03.py](CaseStudy_6.2_01-03.py)

Before running these scripts you will need to follow the instructions in the book to download the example resumes and place them inside the folder:
* [data/resumes](data/resumes)

**NOTE:** The model used in the book has been deprecated so you will need to experiment with alternatives.

In [ ]:
import pandas as pd
import argparse
import fitz
import docx
import os

## 01 - Extract text from documents

Our resumes come in different document formats that need to be converted to pure text before processing

In [ ]:
def extract_text_from_docx(filename):
    doc = docx.Document(filename)
    text_list = []
    for para in doc.paragraphs:
        text_list.append(para.text)
    return '\n'.join(text_list)

In [ ]:
def extract_text_from_pdf(filename):
    with fitz.open(filename) as doc:
        text_list = []
        for page in doc:
            #text_list.append(page.get_text())      ### THIS CHANGE WAS REQUIRED FOR CERTAIN VERSIONS OF fitz
            text_list.append(page.get_textpage().extractText())
        return '\n'.join(text_list)

In [ ]:
def main(dir_path, output_path):
    results = pd.DataFrame()
    for entry in os.scandir(dir_path):
        text = ""
        if entry.is_file():
            if entry.name.endswith(".docx"):
                text = extract_text_from_docx(entry.path)
            if entry.name.endswith(".pdf"):
                text = extract_text_from_pdf(entry.path)
        if text != "":
            record = {"file":entry.name, "text":text}
            results = pd.concat([results,pd.DataFrame([record])], ignore_index=True)
    results.to_csv(output_path, index=False)

### Execute the extraction of text

In the book this was built as a command line application. Here you can simply set your variables and run.

In [ ]:
in_dir = "data/resumes"
out_file = "data/resumes.csv"
main(in_dir, out_file)

## 02 - Extract Selection Criteria

Now that the resumes are all inside a single file we can process them by extracting the specific selection criteria required by the job description.

We created a JSON file for this task called [job_listing.json](job_listing.json) that outlines what these selection criteria are.
This file will be read and used in the code below.


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

class SelectionCriteria(BaseModel):
    education: str = Field(description="A summary of education and credentials.")
    experience: str = Field(description="A summary of relevant work experience.")
    skills: str = Field(description="A summary of key skills.")

parser = JsonOutputParser(pydantic_object=SelectionCriteria)

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
   template = """
     Your job is to extract selection criteria from a job candidate's resume.
     The complete text of the resume will be provided, you need to identify the
     parts of the resume that contain descriptions of their education, work experience
     and key skills. Each of these three elements should be summarised and returned
     in a separate variables according to the instructions below.
      ---
     Here is the resume for you to process:
     ---
     {resume}.
     ---
     {format_instructions}
     """,
   input_variables=["resume"],
   partial_variables={"format_instructions":parser.get_format_instructions()}
)

In [ ]:
def extract_criteria(chain, resume):
   results = {}
   criteria = ["education", "experience", "skills"]
   try:
       response = chain.invoke({"resume":resume})
       for k in criteria:
           if k in response:
               results[k] = response[k]
           else:
               results[k] = ""
   except:
       for k in criteria:
           results[k] = ""
   return results

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model="gemini-1.5-pro")

In [ ]:
resumes = pd.read_csv("data/resumes.csv")
processed = pd.DataFrame()

In [ ]:
chain = prompt | model | parser

for index,row in resumes.iterrows():
    criteria = extract_criteria(chain, row['text'])
    record = {"file":row['file'],
              "education":criteria["education"],
              "experience":criteria["experience"],
              "skills":criteria["skills"]
    }
    processed = pd.concat([processed, pd.DataFrame([record])], ignore_index=True)

In [ ]:
processed.to_csv("data/resumes_processed.csv", index=False)

## 03 Evaluate Selection Criteria

No that each resume has been converted to a response to the individual selection criteria we can apply an evaluation pattern.

In [ ]:
class CriteriaMatch(BaseModel):
    result: bool = Field(description="Binary indicator for selection criteria match")
    score: int = Field(description="A numeric value between 0 and 100 indicating the strength of the match.")
    reason: str = Field(description="A text explanation for the score provided.")

parser = JsonOutputParser(pydantic_object=CriteriaMatch)

In [ ]:
prompt = PromptTemplate(
   template = """
     Your job is to evaluate whether a job candidate matches a particular selection criteria
     based on an extract from their resume. We will provide a description of the selection criteria
     and the relevant excerpt from their resume. You should return a boolean variable to indicate
     if they match, and a score between 0 and 100 baed on the strength of the match. In addition,
     provide a reason that explains the score that was given.
     Here is an example:
     ---
     criteria: Education : A degree in either Computer Science or Engineering.
     resume: I am studying bachelors degree in computer science and I will complete it in 2027.
     result: False
     score: 40
     reason: This candidate is studying the required degree, but they have not yet completed it.
     ---
     criteria: Experience: Candidate should have at least three years experience building iOS apps.
     resume: Exmployer ACME corp: iOS developer 2020-2022, Android developer 2022-2025.
     result: True
     score: 80
     reason: The candidate has done approximately 3 years of iOS development, but it is not recent experience.
     --------
     Here is data for you to process:
     ---
     criteria: {criteria}
     resume: {resume}
     ---
     {format_instructions}
     """,
   input_variables=["criteria", "resume"],
   partial_variables={"format_instructions":parser.get_format_instructions()}
)

In [ ]:
def evaluate_criteria(chain, criteria, resume):
   results = {}
   fields = ["result", "score", "reason"]
   try:
       response = chain.invoke({"criteria":criteria, "resume":resume})
       for k in fields:
           if k in response:
               results[k] = response[k]
           else:
               results[k] = ""
   except:
       for k in fields:
           results[k] = ""
   return results

In [ ]:
import json

with open("job_listing.json", 'r') as f:
    criteria = json.load(f)

In [ ]:
resumes = pd.read_csv("data/resumes_processed.csv")
scored = pd.DataFrame()

In [ ]:
model = ChatGoogleGenerativeAI(model="gemini-1.5-pro")
chain = prompt | model | parser

In [ ]:
for index,row in resumes.iterrows():
    record = {"file":row['file']}
    for k in criteria.keys():
        crit = criteria[k]
        match = evaluate_criteria(chain, crit, row[k])
        prefix = k[0:3]
        for r in match.keys():
            record[prefix+"_"+r] = match[r]
    scored = pd.concat([scored, pd.DataFrame([record])], ignore_index=True)

scored.to_csv("data/resumes_scored.csv", index=False)

### Visualise

In [ ]:
import matplotlib.pyplot as plt

plt.hist(scored['total'])
plt.xlabel("Total Score")
plt.ylabel("Number of Candidates")
plt.title("Distribution of Selection Criteria Scores")
plt.show()

In [ ]:
max_score =  scored['total'].max()
best_candidate = scored.index[ scored['total'] == max_score]
best = scored.loc[best_candidate.values[0],:]

for col in ['edu_reason', 'ski_reason', 'exp_reason']:
    best[col] = best[col][:55]

print(best.to_markdown())